# Numerical variability of fMRI graph measures

## Figures 2 and 3: local and global graph metrics

This notebook reproduces the two figures that compare:

- **numerical variability (NV):** variability across Monte Carlo Arithmetic (MCA) repetitions;
- **population variability (PV):** variability across participants; and
- **NPVR:** the ratio $\sigma_{\mathrm{num}} / \sigma_{\mathrm{pop}}$.

Results are shown for the combined Parkinson's disease and healthy-control population, and NPVR is also summarized separately for the two diagnostic groups. The analysis uses session 1, right-to-left acquisition (`acq-RL`), 10 MCA repetitions, and a 100-region Schaefer parcellation.

Running the notebook writes `Figures/Fig2_local_metrics.{png,pdf}` and `Figures/Fig3_global_metrics.{png,pdf}`. Committed PNG previews make the results visible in GitHub's static notebook viewer.

## Reproducibility and data layout

Python 3.11+ dependencies: `numpy>=2`, `pandas>=2`, and `matplotlib>=3.7`. Install them with:

```bash
python -m pip install "numpy>=2" "pandas>=2" "matplotlib>=3.7"
```

Set `FMRI_DATA_ROOT` to the MCA directory containing the threshold folders. In this project the notebook discovers `overlap/allbatches/Allpop/Absolute_thresholding/MCA` automatically:

```text
$FMRI_DATA_ROOT/
├── table_corr0.05/
│   ├── Result_tableWConf_batch_1.pkl
│   ├── ...
│   └── Result_tableWConf_batch_hc.pkl
├── table_corr0.1/
├── ...
└── table_corr0.5/
```

Both the combined-population and separate PD/HC estimates are calculated directly from these pickle files; no additional summary CSV files are required. Subject exclusions below correspond to failed preprocessing/QC. Variances use NumPy's population convention (`ddof=0`), matching the original analysis.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd


ANALYSIS_REPOSITORY_RELATIVE = Path(
    "overlap/allbatches/Allpop/pickles/"
    "Numerical-Variability-of-functional-MRI-Graph-Measures"
)
THRESHOLDS = (0.05, 0.1, 0.2, 0.3, 0.4, 0.5)
BATCHES = (1, 2, 3, 4, 5)
SESSION = "1"
ACQUISITION = "acq-RL"
EXCLUDED_ACQUISITIONS = {"acq-RLsplit1", "acq-LRsplit1"}

LOCAL_METRICS = {
    "degree": ("degree_centralities", "Degree centrality"),
    "betweenness": ("betweenness_centralities", "Betweenness centrality"),
    "eigenvector": ("eigenvector_centralities", "Eigenvector centrality"),
    "clustering": ("clustering_coefficients", "Clustering coefficient"),
}
GLOBAL_METRICS = {
    "small_worldness": ("small_worldness", "Small-worldness"),
    "average_shortest_path_length": (
        "avg_shortest_path_length",
        "Average shortest path length",
    ),
}
METRICS = {**LOCAL_METRICS, **GLOBAL_METRICS}
#Failed preprocessing subjects
def load_subject_exclusions(repository_root: Path) -> tuple[set[str], set[str]]:
    """Load restricted QC exclusions from a local, ignored JSON file."""
    configured = os.environ.get("PPMI_SUBJECT_EXCLUSIONS_FILE")
    candidates = []
    if configured:
        candidates.append(Path(configured).expanduser())
    candidates.append(repository_root / "data" / "subject_exclusions.json")
    candidates.extend(
        candidate / "data" / "subject_exclusions.json"
        for candidate in (Path.cwd(), *Path.cwd().parents)
    )

    for candidate in candidates:
        if candidate.is_file():
            payload = json.loads(candidate.read_text(encoding="utf-8"))
            return set(payload.get("pd", [])), set(payload.get("hc", []))

    raise FileNotFoundError(
        "Set PPMI_SUBJECT_EXCLUSIONS_FILE or create the ignored local file "
        "data/subject_exclusions.json with 'pd' and 'hc' arrays."
    )


def find_repository_root(start: Path | None = None) -> Path:
    """Find the nearest parent that contains this repository's `.git` entry."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (
            candidate / "Fig2-3-GraphMetrics-NPVR"
        ).is_dir():
            return candidate
    for base in (start, *start.parents):
        candidate = base / ANALYSIS_REPOSITORY_RELATIVE
        if (candidate / ".git").exists():
            return candidate
    return start


def has_threshold_data(candidate: Path) -> bool:
    """Return whether a directory contains the expected MCA inputs."""
    return any(
        (candidate / folder).is_dir()
        for folder in ("table_corr0.05", "table_correther0.05")
    )


def find_data_root(repository_root: Path) -> Path:
    """Resolve the MCA data root from the environment or project layout."""
    if configured := os.environ.get("FMRI_DATA_ROOT"):
        candidate = Path(configured).expanduser().resolve()
        if not candidate.is_dir():
            raise FileNotFoundError(f"FMRI_DATA_ROOT does not exist: {candidate}")
        if not has_threshold_data(candidate):
            raise FileNotFoundError(
                f"FMRI_DATA_ROOT has no table_corr* folders: {candidate}"
            )
        return candidate

    candidates = []
    for base in (repository_root, *repository_root.parents):
        candidates.extend([
            base,
            base / "Absolute_thresholding/MCA",
            base / "overlap/allbatches/Allpop/Absolute_thresholding/MCA",
        ])
    for candidate in candidates:
        if has_threshold_data(candidate):
            return candidate

    raise FileNotFoundError(
        "Could not locate table_corr0.05. Set FMRI_DATA_ROOT to the MCA "
        "directory described above."
    )


REPOSITORY_ROOT = find_repository_root()
PD_SUBJECTS_TO_REMOVE, HC_SUBJECTS_TO_REMOVE = load_subject_exclusions(
    REPOSITORY_ROOT
)
DATA_ROOT = find_data_root(REPOSITORY_ROOT)
FIGURE_DIR = REPOSITORY_ROOT / "Fig2-3-GraphMetrics-NPVR"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository: {REPOSITORY_ROOT}")
print(f"Data:       {DATA_ROOT}")
print(f"Figures:    {FIGURE_DIR}")

## 1. Load and quality-control the graph measures

For each threshold, the five Parkinson's disease batches are concatenated and combined with the healthy-control batch. Both groups use the same `WConf` input condition. This fixes an earlier copy/paste error that duplicated the healthy-control data at thresholds 0.05–0.4 and mixed input conditions at threshold 0.5.

In [ ]:
def threshold_directory(threshold: float) -> Path:
    for prefix in ("table_corr", "table_correther"):
        directory = DATA_ROOT / f"{prefix}{threshold}"
        if directory.is_dir():
            return directory
    raise FileNotFoundError(f"No data folder found for threshold {threshold}")


def read_population_data(threshold: float) -> dict[str, pd.DataFrame]:
    """Load and QC the WConf inputs for one graph-construction threshold."""
    directory = threshold_directory(threshold)
    pd_frames = [
        pd.read_pickle(directory / f"Result_tableWConf_batch_{batch}.pkl")
        for batch in BATCHES
    ]
    pd_data = pd.concat(pd_frames, ignore_index=True)
    hc_data = pd.read_pickle(directory / "Result_tableWConf_batch_hc.pkl")

    pd_data = pd_data.loc[~pd_data["subject"].isin(PD_SUBJECTS_TO_REMOVE)].copy()
    hc_data = hc_data.loc[~hc_data["subject"].isin(HC_SUBJECTS_TO_REMOVE)].copy()
    combined = pd.concat([pd_data, hc_data], ignore_index=True)

    return {"pd": pd_data, "hc": hc_data, "combined": combined}


population_data = {
    threshold: read_population_data(threshold) for threshold in THRESHOLDS
}

sample_sizes = pd.DataFrame(
    {
        "threshold": threshold,
        "PD": frames["pd"]["subject"].nunique(),
        "HC": frames["hc"]["subject"].nunique(),
        "combined": frames["combined"]["subject"].nunique(),
    }
    for threshold, frames in population_data.items()
).set_index("threshold")
sample_sizes

In [ ]:
population_data

## 2. Estimate numerical and population variability

Let $X_{s,r,m,k}$ denote graph metric $m$ for subject $s$, region $r$, and MCA repetition $k$.

Numerical variability is first estimated within each subject:

$$
V^{\mathrm{num}}_{s,r,m} = \operatorname{Var}_{k}(X_{s,r,m,k}),
\qquad
\sigma^{\mathrm{num}}_{r,m} =
\sqrt{\operatorname{mean}_{s}\!\left(V^{\mathrm{num}}_{s,r,m}\right)}.
$$

Population variability is estimated across subjects within each repetition:

$$
V^{\mathrm{pop}}_{k,r,m} = \operatorname{Var}_{s}(X_{s,r,m,k}),
\qquad
\sigma^{\mathrm{pop}}_{r,m} =
\sqrt{\operatorname{mean}_{k}\!\left(V^{\mathrm{pop}}_{k,r,m}\right)}.
$$

Local metrics yield one value per atlas region; global metrics yield one scalar value.

In [ ]:
def metric_values(group: pd.DataFrame, source_column: str) -> np.ndarray:
    """Convert a graph-metric column to a numeric observation array."""
    values = []
    for value in group[source_column]:
        if isinstance(value, dict):
            value = list(value.values())
        values.append(np.asarray(value, dtype=float))
    return np.asarray(values, dtype=float)


def calculate_group_variances(
    data: pd.DataFrame,
    group_columns: list[str],
    *,
    unique_subjects: bool = False,
    minimum_observations: int = 1,
) -> pd.DataFrame:
    """Calculate metric variances within the requested grouping dimensions."""
    filtered = data.loc[~data["acquisition"].isin(EXCLUDED_ACQUISITIONS)].copy()
    rows = []

    for keys, group in filtered.groupby(group_columns, sort=True):
        if unique_subjects:
            group = group.drop_duplicates("subject")
        if len(group) < minimum_observations:
            continue

        keys = keys if isinstance(keys, tuple) else (keys,)
        result = dict(zip(group_columns, keys))
        for metric, (source_column, _) in METRICS.items():
            result[metric] = np.var(metric_values(group, source_column), axis=0)
        rows.append(result)

    return pd.DataFrame(rows)


def estimate_variability(data: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Estimate within-subject numerical and between-subject variances."""
    numerical = calculate_group_variances(
        data,
        ["subject", "session", "acquisition"],
        minimum_observations=2,
    )
    population = calculate_group_variances(
        data,
        ["repetition", "session", "acquisition"],
        unique_subjects=True,
        minimum_observations=2,
    )
    return {"numerical": numerical, "population": population}


def root_mean_variance(variance_data: pd.DataFrame) -> dict[str, np.ndarray | float]:
    """Convert repeated variance estimates to root-mean variance."""
    selected = variance_data.loc[
        (variance_data["session"].astype(str) == SESSION)
        & (variance_data["acquisition"] == ACQUISITION)
    ]
    if selected.empty:
        raise ValueError(f"No rows found for session={SESSION}, acquisition={ACQUISITION}")

    result = {}
    for metric in METRICS:
        stacked = np.stack(selected[metric].to_numpy())
        value = np.sqrt(np.mean(stacked, axis=0))
        result[metric] = float(value) if value.ndim == 0 else value
    return result

In [ ]:
variance_estimates = {
    threshold: estimate_variability(frames["combined"])
    for threshold, frames in population_data.items()
}

variability = {
    threshold: {
        kind: root_mean_variance(variance_frame)
        for kind, variance_frame in estimates.items()
    }
    for threshold, estimates in variance_estimates.items()
}

print(f"Computed NV and PV at {len(variability)} thresholds.")

## 3. Compute NPVR

For each region and graph metric, the numerical-to-population variability ratio is

$$
\operatorname{NPVR}_{r,m} =
\frac{\sigma^{\mathrm{num}}_{r,m}}{\sigma^{\mathrm{pop}}_{r,m}}.
$$

The combined-population, PD-only, and HC-only NPVR values are all computed directly from the MCA pickle files loaded above.

In [ ]:
def safe_ratio(numerator, denominator):
    """Divide elementwise and represent zero denominators as missing values."""
    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.divide(
            numerator,
            denominator,
            out=np.full_like(numerator, np.nan, dtype=float),
            where=denominator != 0,
        )
    return float(ratio) if ratio.ndim == 0 else ratio


combined_npvr = {
    threshold: {
        metric: safe_ratio(
            estimates["numerical"][metric], estimates["population"][metric]
        )
        for metric in METRICS
    }
    for threshold, estimates in variability.items()
}


group_variability = {
    group: {
        threshold: {
            kind: root_mean_variance(variance_frame)
            for kind, variance_frame in estimate_variability(
                population_data[threshold][group]
            ).items()
        }
        for threshold in THRESHOLDS
    }
    for group in ("pd", "hc")
}

group_npvr = {
    group: {
        threshold: {
            metric: safe_ratio(
                estimates["numerical"][metric],
                estimates["population"][metric],
            )
            for metric in METRICS
        }
        for threshold, estimates in thresholds.items()
    }
    for group, thresholds in group_variability.items()
}

In [ ]:
def regional_mean(value) -> float:
    return float(np.nanmean(np.asarray(value, dtype=float)))


summary_rows = []
for threshold in THRESHOLDS:
    for metric, (_, label) in METRICS.items():
        summary_rows.append(
            {
                "threshold": threshold,
                "metric": label,
                "NV": regional_mean(variability[threshold]["numerical"][metric]),
                "PV": regional_mean(variability[threshold]["population"][metric]),
                "NPVR combined": regional_mean(combined_npvr[threshold][metric]),
                "NPVR PD": regional_mean(group_npvr["pd"][threshold][metric]),
                "NPVR HC": regional_mean(group_npvr["hc"][threshold][metric]),
            }
        )

summary = pd.DataFrame(summary_rows)
summary.round(4)

## 4. Figure 2 — local graph metrics

Boxes show the distribution across the 100 atlas regions. Stars and dashed lines use the right-hand axis and show the regional mean NPVR for the combined population, PD, and healthy controls.

In [ ]:
COLORS = {
    "numerical": "#A59F9F",
    "population": "#2B2A2A",
    "combined": "#000000",
    "pd": "red",
    "hc": "royalblue",
}


def save_figure(figure: plt.Figure, stem: str) -> None:
    """Save a figure in publication and preview formats."""
    figure.savefig(FIGURE_DIR / f"{stem}.pdf", bbox_inches="tight")
    figure.savefig(FIGURE_DIR / f"{stem}.png", dpi=180, bbox_inches="tight")


def add_original_caption(figure: plt.Figure, y_positions: tuple[float, float, float]) -> None:
    """Add the original three-row PD+HC, PD, and HC figure caption."""
    combined_handles = [
        Patch(facecolor=COLORS["numerical"], edgecolor="black", label=r"PD+HC: NV ($\sigma_{num}$)"),
        Patch(facecolor=COLORS["population"], edgecolor="black", label=r"PV ($\sigma_{pop}$)"),
        Line2D([0], [0], color=COLORS["combined"], marker="*", linestyle="None", markersize=11, label="Mean NPVR"),
    ]
    combined_caption = figure.legend(
        handles=combined_handles, loc="lower center",
        bbox_to_anchor=(0.5, y_positions[0]), ncol=3, frameon=False,
    )
    figure.add_artist(combined_caption)

    for group, label, y in (("pd", "PD", y_positions[1]), ("hc", "HC", y_positions[2])):
        caption = figure.legend(
            handles=[Line2D(
                [0], [0], color=COLORS[group], marker="*", linestyle="None",
                markersize=11, label=f"{label}: Mean NPVR",
            )],
            loc="lower center", bbox_to_anchor=(0.5, y), frameon=False,
        )
        figure.add_artist(caption)


def plot_local_metrics() -> plt.Figure:
    figure, axes = plt.subplots(4, 1, figsize=(12, 18), sharex=True)
    x = np.arange(len(THRESHOLDS), dtype=float)

    for axis, (metric, (_, label)) in zip(axes, LOCAL_METRICS.items()):
        nv = [variability[t]["numerical"][metric] for t in THRESHOLDS]
        pv = [variability[t]["population"][metric] for t in THRESHOLDS]

        for values, positions, color in (
            (nv, x - 0.13, COLORS["numerical"]),
            (pv, x + 0.13, COLORS["population"]),
        ):
            boxes = axis.boxplot(
                values,
                positions=positions,
                widths=0.22,
                patch_artist=True,
                showfliers=False,
                manage_ticks=False,
            )
            for box in boxes["boxes"]:
                box.set(facecolor=color, edgecolor="black", linewidth=0.8)
            for element in ("whiskers", "caps", "medians"):
                plt.setp(boxes[element], color="black", linewidth=0.8)
            for position, region_values in zip(positions, values):
                jitter = np.linspace(-0.035, 0.035, len(region_values))
                axis.scatter(
                    position + jitter, region_values, s=7, color=color,
                    alpha=0.55, edgecolors="none", zorder=2,
                )

        ratio_axis = axis.twinx()
        ratio_series = {
            "combined": [regional_mean(combined_npvr[t][metric]) for t in THRESHOLDS],
            "pd": [regional_mean(group_npvr["pd"][t][metric]) for t in THRESHOLDS],
            "hc": [regional_mean(group_npvr["hc"][t][metric]) for t in THRESHOLDS],
        }
        for group, values in ratio_series.items():
            ratio_axis.plot(
                x,
                values,
                color=COLORS[group],
                marker="*",
                markersize=11,
                linewidth=1.8,
                linestyle="--",
            )

        axis.set_title(label, fontweight="bold", fontsize=16)
        axis.set_ylabel("Numerical or population variability", fontsize=14)
        ratio_axis.set_ylabel("Mean NPVR", fontsize=14)

        axis.tick_params(axis="both", labelsize=14)
        ratio_axis.tick_params(axis="y", labelsize=14)

        axis.grid(axis="y", color="0.9", linewidth=0.8)

    axes[-1].set_xticks(x, [f"T = {threshold:g}" for threshold in THRESHOLDS])
    axes[-1].set_xlabel("Threshold Values", fontweight="bold")
    add_original_caption(figure, (0.065, 0.038, 0.011))
    figure.tight_layout(rect=(0, 0.105, 1, 1))
    return figure


figure2 = plot_local_metrics()
save_figure(figure2, "Fig2_local_metrics")
figure2.savefig(
    "Fig2_local_metrics.tif",
    dpi=600,
    bbox_inches="tight",
    pil_kwargs={"compression": "tiff_lzw"}
)  # main-figure upload
plt.show()

## 5. Figure 3 — global graph metrics

NV and PV use the left-hand axis. Mean NPVR for the combined population, PD, and healthy controls uses the right-hand axis.

## 5. Figure 3 — global graph metrics

NV and PV use the left-hand axis. Mean NPVR for the combined population, PD, and healthy controls uses the right-hand axis.

In [ ]:
def plot_global_metrics() -> plt.Figure:
    figure, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    x = np.arange(len(THRESHOLDS), dtype=float)

    for axis, (metric, (_, label)) in zip(axes, GLOBAL_METRICS.items()):
        nv = [variability[t]["numerical"][metric] for t in THRESHOLDS]
        pv = [variability[t]["population"][metric] for t in THRESHOLDS]
        axis.plot(
            x - 0.04, nv, color=COLORS["numerical"], marker="o",
            markeredgecolor="black", markersize=8, linestyle="None",
        )
        axis.plot(
            x + 0.04, pv, color=COLORS["population"], marker="o",
            markeredgecolor="black", markersize=8, linestyle="None",
        )

        ratio_axis = axis.twinx()
        ratio_series = {
            "combined": [combined_npvr[t][metric] for t in THRESHOLDS],
            "pd": [group_npvr["pd"][t][metric] for t in THRESHOLDS],
            "hc": [group_npvr["hc"][t][metric] for t in THRESHOLDS],
        }
        for group, values in ratio_series.items():
            ratio_axis.plot(
                x,
                values,
                color=COLORS[group],
                marker="*",
                markersize=11,
                linewidth=1.8,
                linestyle="--",
            )

        axis.set_title(label, fontweight="bold",fontsize=16)
        axis.set_ylabel("Numerical or population variability", fontsize=14)
        ratio_axis.set_ylabel("Mean NPVR", fontsize=14)
        axis.tick_params(axis="both", labelsize=14)
        ratio_axis.tick_params(axis="y", labelsize=14)
        axis.grid(axis="y", color="0.9", linewidth=0.8)

    axes[-1].set_xticks(x, [f"T = {threshold:g}" for threshold in THRESHOLDS])
    axes[-1].set_xlabel("Threshold", fontweight="bold")
    add_original_caption(figure, (0.066, 0.01, 0.031))
    figure.tight_layout(rect=(0, 0.18, 1, 1))
    figure.subplots_adjust(
    hspace=0.2,  # increase this value for more space
    bottom=0.18,
    top=0.95,
)
    return figure


figure3 = plot_global_metrics()
save_figure(figure3, "Fig3_global_metrics")
figure3.savefig(
    "Fig3_global_metrics.tif",
    dpi=600,
    bbox_inches="tight",
    pil_kwargs={"compression": "tiff_lzw"}
)  # main-figure upload
plt.show()



## Interpretation guide

- **NPVR < 1:** population variability is larger than numerical variability.
- **NPVR = 1:** the two variability sources have the same magnitude.
- **NPVR > 1:** numerical variability is larger than population variability.

The figures summarize relative magnitudes; they do not provide inferential tests or uncertainty intervals. See the accompanying analysis notebooks for group comparisons and statistical inference.